<a href="https://colab.research.google.com/github/AKSHAYA-1412/GEN-AI-TRAINING/blob/main/WEEK_1_ASSIGNMENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q openai faiss-cpu pypdf numpy

In [ ]:
import os
import getpass
import numpy as np
import faiss
from google.colab import files
from pypdf import PdfReader
from openai import OpenAI

In [ ]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OpenAI API Key: ")
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

Enter OpenAI API Key: ··········


In [ ]:
uploaded = files.upload()

Saving RAG.pdf to RAG.pdf


In [ ]:
documents = []

for filename in uploaded.keys():
    text = ""

    if filename.endswith(".pdf"):
        reader = PdfReader(filename)
        for page in reader.pages:
            extracted = page.extract_text()
            if extracted:
                text += extracted + "\n"

    elif filename.endswith(".txt") or filename.endswith(".md"):
        with open(filename, "r", encoding="utf-8") as f:
            text = f.read()

    if text.strip():
        documents.append({"source": filename, "text": text})

print("Loaded Documents:", len(documents))

Loaded Documents: 1


In [ ]:
def chunk_text(text, chunk_size=1000, overlap=200):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap

    return chunks

In [ ]:
texts = [chunk["text"] for chunk in all_chunks]

response = client.embeddings.create(
    model="text-embedding-3-small",
    input=texts
)

embeddings = [item.embedding for item in response.data]
embedding_matrix = np.array(embeddings).astype("float32")

print("Embeddings Created:", embedding_matrix.shape)

Embeddings Created: (124, 1536)


In [ ]:
dimension = embedding_matrix.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embedding_matrix)

print("FAISS Index Ready!")

FAISS Index Ready!


In [ ]:
while True:
    query = input("\nAsk a question (type exit to stop): ")

    if query.lower() == "exit":
        print("Goodbye bro")
        break

    # Create query embedding
    q_embed = client.embeddings.create(
        model="text-embedding-3-small",
        input=[query]
    )

    query_vector = np.array([q_embed.data[0].embedding]).astype("float32")

    # Search top 5 instead of 3
    distances, indices = index.search(query_vector, 5)

    retrieved_chunks = []
    for idx in indices[0]:
        if idx != -1:
            retrieved_chunks.append(all_chunks[idx]["text"])

    context = "\n\n".join(retrieved_chunks)

    prompt = f"""
You are a smart Knowledge Assistant.

Answer the user's question using the provided context.
If the context partially contains the answer, explain clearly.
Only say 'I couldn't find the answer...' if the context is completely unrelated.

Context:
{context}

Question: {query}

Helpful Answer:
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You answer clearly from retrieved document context."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )

    print("\nAnswer:\n")
    print(response.choices[0].message.content)


Ask a question (type exit to stop): What is RAG?

🤖 Answer:

RAG stands for Retrieval-Augmented Generation. It is a process that combines retrieval of information from high-quality text corpora with generative models to enhance the quality of answers provided by knowledge question-answering systems. The effectiveness of RAG systems is significantly influenced by the quality of document parsing and chunking, particularly when dealing with professional documents that are often stored in PDFs.

Ask a question (type exit to stop): What is CHucking?

🤖 Answer:

The context provided does not contain any information about "CHucking." It primarily discusses issues related to data retrieval, token limits, and the structure of tables in the context of a PDF parser. Therefore, I couldn't find the answer to your question about CHucking.

Ask a question (type exit to stop): CHunking?

🤖 Answer:

The context provided discusses issues related to chunking in data retrieval, particularly in the contex